# Emoji Toxicity Detection — Data Analysis & Results

This notebook explores the knowledge base and visualizes evaluation results.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

DATA_DIR = Path("..") / "data"
KB_PATH = DATA_DIR / "processed" / "knowledge_base.jsonl"
RESULTS_PATH = DATA_DIR / "results" / "eval_results.json"

## 1. Knowledge Base Analysis

In [ ]:
# Load KB
entries = []
if KB_PATH.exists():
    with open(KB_PATH) as f:
        entries = [json.loads(line) for line in f if line.strip()]
    print(f"Knowledge base: {len(entries)} entries")
else:
    print("KB not built yet. Run: python -m scripts.build_knowledge_base")

if entries:
    df = pd.DataFrame(entries)
    print(f"\nRisk category distribution:")
    print(df['risk_category'].value_counts())
    
    print(f"\nSources:")
    all_sources = [s for sources in df['sources'] for s in sources]
    print(Counter(all_sources).most_common())

In [ ]:
# Visualize risk categories
if entries:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    df['risk_category'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Knowledge Base — Risk Category Distribution')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 2. Evaluation Results

In [ ]:
# Load evaluation results
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    
    results_df = pd.DataFrame(results)
    print(results_df[['name', 'n_samples', 'accuracy', 'precision', 'recall', 'f1_macro', 'auroc']].to_string(index=False))
    
    # Bar chart comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_macro']
    x = range(len(results_df))
    width = 0.2
    
    for i, metric in enumerate(metrics_to_plot):
        vals = results_df[metric].tolist()
        ax.bar([xi + i * width for xi in x], vals, width, label=metric)
    
    ax.set_xticks([xi + 1.5 * width for xi in x])
    ax.set_xticklabels(results_df['name'])
    ax.set_ylim(0, 1)
    ax.legend()
    ax.set_title('Evaluation: Method Comparison')
    plt.tight_layout()
    plt.show()
else:
    print("No results yet. Run: python -m scripts.evaluate")